# Notebook Lengkap: Deteksi Plagiarisme Semantik
## Studi Komparatif Multilingual Sentence-BERT dan IndoBERT pada Teks Bahasa Indonesia

**Tahapan:**
- EDA & Preprocessing
- Model Comparison (TF-IDF, SBERT, IndoBERT)
- Evaluation & Visualization

**Dataset:** MSRP Indonesia (HuggingFace)

In [ ]:
!pip install -q scikit-learn sentence-transformers transformers torch pandas matplotlib numpy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import re
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('data', exist_ok=True)
os.makedirs('assets', exist_ok=True)
print("Setup complete.")

# 1. EDA & Preprocessing

In [ ]:
train_url = 'https://media.githubusercontent.com/media/jakartaresearch/hf-datasets/main/msrp/id_train.csv'
val_url = 'https://media.githubusercontent.com/media/jakartaresearch/hf-datasets/main/msrp/id_test.csv'

if os.path.exists('data/id_msrp_train.csv'):
    print('Dataset sudah tersedia, loading dari lokal...')
    msrp_train = pd.read_csv('data/id_msrp_train.csv')
    msrp_val = pd.read_csv('data/id_msrp_val.csv')
else:
    print('Downloading dataset dari HuggingFace...')
    msrp_train = pd.read_csv(train_url)
    msrp_val = pd.read_csv(val_url)
    msrp_train.to_csv('data/id_msrp_train.csv', index=False)
    msrp_val.to_csv('data/id_msrp_val.csv', index=False)

print(f"Train: {len(msrp_train)} rows")
print(f"Validation: {len(msrp_val)} rows")
print(f"Columns: {msrp_train.columns.tolist()}")

In [ ]:
print("Label distribution (train):")
print(msrp_train['label'].value_counts())
print("\nLabel distribution (val):")
print(msrp_val['label'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

msrp_train['label'].value_counts().plot(kind='bar', ax=axes[0], color=['skyblue', 'salmon'])
axes[0].set_title('MSRP Train - Label Distribution')
axes[0].set_xlabel('Label (0=Bukan Plagiarisme, 1=Plagiarisme)')
axes[0].set_ylabel('Count')

msrp_val['label'].value_counts().plot(kind='bar', ax=axes[1], color=['skyblue', 'salmon'])
axes[1].set_title('MSRP Validation - Label Distribution')
axes[1].set_xlabel('Label (0=Bukan Plagiarisme, 1=Plagiarisme)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('assets/label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: assets/label_distribution.png")

In [ ]:
print("=== CONTOH PLAGIARISME SEMANTIK (Label=1) ===")
pos = msrp_train[msrp_train['label'] == 1].head(3)
for _, row in pos.iterrows():
    print(f"Teks 1: {row['sentence1']}")
    print(f"Teks 2: {row['sentence2']}")
    print("---")

print("\n=== CONTOH BUKAN PLAGIARISME (Label=0) ===")
neg = msrp_train[msrp_train['label'] == 0].head(3)
for _, row in neg.iterrows():
    print(f"Teks 1: {row['sentence1']}")
    print(f"Teks 2: {row['sentence2']}")
    print("---")

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

msrp_train['sentence1_clean'] = msrp_train['sentence1'].apply(clean_text)
msrp_train['sentence2_clean'] = msrp_train['sentence2'].apply(clean_text)
msrp_val['sentence1_clean'] = msrp_val['sentence1'].apply(clean_text)
msrp_val['sentence2_clean'] = msrp_val['sentence2'].apply(clean_text)

print("Contoh setelah cleaning:")
print(f"Original: {msrp_train.iloc[0]['sentence1']}")
print(f"Cleaned:  {msrp_train.iloc[0]['sentence1_clean']}")

In [ ]:
# Hitung statistik teks
train_words1 = msrp_train['sentence1_clean'].str.split().str.len()
train_words2 = msrp_train['sentence2_clean'].str.split().str.len()

print("Statistik panjang teks (train):")
print(f"  Teks 1 - Mean: {train_words1.mean():.1f}, Std: {train_words1.std():.1f}, Max: {train_words1.max()}")
print(f"  Teks 2 - Mean: {train_words2.mean():.1f}, Std: {train_words2.std():.1f}, Max: {train_words2.max()}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(train_words1, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
ax1.set_title('Distribusi Panjang Teks 1')
ax1.set_xlabel('Jumlah Kata')
ax1.set_ylabel('Frekuensi')
ax2.hist(train_words2, bins=50, alpha=0.7, color='salmon', edgecolor='black')
ax2.set_title('Distribusi Panjang Teks 2')
ax2.set_xlabel('Jumlah Kata')
ax2.set_ylabel('Frekuensi')
plt.tight_layout()
plt.savefig('assets/text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: assets/text_length_distribution.png")

In [ ]:
# Hitung overlap kata
def word_overlap(t1, t2):
    s1 = set(t1.split())
    s2 = set(t2.split())
    if not s1 or not s2:
        return 0
    return len(s1 & s2) / len(s1 | s2)

msrp_train['overlap'] = msrp_train.apply(lambda r: word_overlap(r['sentence1_clean'], r['sentence2_clean']), axis=1)

fig, ax = plt.subplots(figsize=(8, 4))
for label in [0, 1]:
    subset = msrp_train[msrp_train['label'] == label]['overlap']
    ax.hist(subset, bins=30, alpha=0.5, label=f'Label {label}', density=True)
ax.set_title('Distribusi Word Overlap by Label')
ax.set_xlabel('Jaccard Similarity (Kata)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig('assets/word_overlap_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: assets/word_overlap_distribution.png")

In [ ]:
# Simpan data clean
msrp_train[['sentence1_clean', 'sentence2_clean', 'label']].to_csv('data/msrp_train_clean.csv', index=False)
msrp_val[['sentence1_clean', 'sentence2_clean', 'label']].to_csv('data/msrp_val_clean.csv', index=False)
print(f"Saved: data/msrp_train_clean.csv ({len(msrp_train)} rows)")
print(f"Saved: data/msrp_val_clean.csv ({len(msrp_val)} rows)")

# 2. Model Comparison

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df_test = pd.read_csv('data/msrp_val_clean.csv')
print(f"Test set: {len(df_test)} pairs")
print(f"Label distribution:\n{df_test['label'].value_counts()}")

pairs = list(zip(df_test['sentence1_clean'], df_test['sentence2_clean']))
y_true = df_test['label'].values
texts1, texts2 = zip(*pairs)

In [ ]:
print("Running TF-IDF...")
vectorizer = TfidfVectorizer(max_features=10000)
all_texts = [t for pair in pairs for t in pair]
vectorizer.fit(all_texts)

vec1 = vectorizer.transform(texts1)
vec2 = vectorizer.transform(texts2)
sims_tfidf = cosine_similarity(vec1, vec2).diagonal()
print(f"TF-IDF done. Mean similarity: {sims_tfidf.mean():.4f}")

In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading SBERT (paraphrase-multilingual-mpnet-base-v2)...")
sbert_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

print("Encoding dengan SBERT...")
emb1_sbert = sbert_model.encode(texts1, batch_size=32, show_progress_bar=True)
emb2_sbert = sbert_model.encode(texts2, batch_size=32, show_progress_bar=True)

sims_sbert = np.array([cosine_similarity([e1], [e2])[0][0] for e1, e2 in zip(emb1_sbert, emb2_sbert)])
print(f"SBERT done. Mean similarity: {sims_sbert.mean():.4f}")

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

print("Loading IndoBERT (indobenchmark/indobert-base-p1)...")
tokenizer = AutoTokenizer.from_pretrained('indobenchmark/indobert-base-p1')
indobert_model = AutoModel.from_pretrained('indobenchmark/indobert-base-p1').to(device)
indobert_model.eval()

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def encode_indobert(texts, batch_size=32):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors='pt')
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            model_output = indobert_model(**encoded)
        batch_emb = mean_pooling(model_output, encoded['attention_mask'])
        embeddings.append(batch_emb.cpu().numpy())
    return np.vstack(embeddings)

print("Encoding dengan IndoBERT...")
emb1_indobert = encode_indobert(texts1)
emb2_indobert = encode_indobert(texts2)

sims_indobert = np.array([cosine_similarity([e1], [e2])[0][0] for e1, e2 in zip(emb1_indobert, emb2_indobert)])
print(f"IndoBERT done. Mean similarity: {sims_indobert.mean():.4f}")

In [ ]:
results = {
    'y_true': y_true,
    'sims_tfidf': sims_tfidf,
    'sims_sbert': sims_sbert,
    'sims_indobert': sims_indobert
}

with open('data/similarities.pkl', 'wb') as f:
    pickle.dump(results, f)

print("Saved: data/similarities.pkl")

# 3. Evaluation & Visualization

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, roc_curve, confusion_matrix

with open('data/similarities.pkl', 'rb') as f:
    results = pickle.load(f)

y_true = results['y_true']
sims = {
    'TF-IDF': results['sims_tfidf'],
    'SBERT': results['sims_sbert'],
    'IndoBERT': results['sims_indobert']
}

print(f"Loaded {len(y_true)} test pairs")

In [ ]:
def find_optimal_threshold(y_true, y_scores):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    optimal_idx = np.argmax(tpr - fpr)
    return thresholds[optimal_idx]

thresholds = {}
for name, scores in sims.items():
    thresh = find_optimal_threshold(y_true, scores)
    thresholds[name] = thresh
    print(f"{name} - Optimal threshold: {thresh:.4f}")

In [ ]:
metrics_list = []

for name, scores in sims.items():
    thresh = thresholds[name]
    y_pred = (scores >= thresh).astype(int)

    metrics = {
        'model': name,
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'auc': roc_auc_score(y_true, scores),
        'threshold': thresh
    }
    metrics_list.append(metrics)

    print(f"\n=== {name} ===")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")

df_metrics = pd.DataFrame(metrics_list)
print("\n=== SUMMARY TABLE ===")
print(df_metrics.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 8))
colors = {'TF-IDF': '#0369a1', 'SBERT': '#7c3aed', 'IndoBERT': '#be123c'}

for name, scores in sims.items():
    fpr, tpr, _ = roc_curve(y_true, scores)
    auc = roc_auc_score(y_true, scores)
    plt.plot(fpr, tpr, color=colors[name], linewidth=2.5, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC = 0.500)')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Perbandingan Model', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('assets/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: assets/roc_curve.png")

In [ ]:
for name, scores in sims.items():
    thresh = thresholds[name]
    y_pred = (scores >= thresh).astype(int)
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(f'Confusion Matrix - {name}', fontsize=13, fontweight='bold')
    plt.colorbar()
    plt.xticks([0, 1], ['Negative', 'Positive'])
    plt.yticks([0, 1], ['Negative', 'Positive'])
    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i, j]), ha='center', va='center', color='red', fontsize=14, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    safe_name = name.lower().replace('-', '').replace(' ', '_')
    plt.savefig(f'assets/confusion_matrix_{safe_name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: assets/confusion_matrix_{safe_name}.png")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_metrics))
width = 0.25
multiplier = 0

for attribute in ['f1', 'precision', 'recall', 'auc']:
    offset = width * multiplier
    bars = ax.bar(x + offset, df_metrics[attribute], width, label=attribute.upper())
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
    multiplier += 1

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(df_metrics['model'])
ax.set_ylabel('Score')
ax.set_title('Perbandingan Metrik Evaluasi per Model', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.legend(loc='upper right')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('assets/metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: assets/metrics_comparison.png")

In [ ]:
df_metrics.to_csv('assets/hasil_evaluasi.csv', index=False)
print("Saved: assets/hasil_evaluasi.csv")

print("\nFile di assets/:")
for f in sorted(os.listdir('assets')):
    print(f"  - assets/{f}")